# Part A

In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
seed = 84735

df = pd.read_csv("Default.csv")
X = df[['income', 'balance']].values
df['default'] = df['default'].map({'Yes': 1, 'No': 0})
y = df['default']

clf = LogisticRegression(random_state = seed).fit(X, y)

# Part B

In [17]:
from sklearn.model_selection import KFold
import numpy as np

kf = KFold(n_splits=10, shuffle=True, random_state=seed)

errors = []

for train_index, test_index in kf.split(X):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    clf = LogisticRegression(random_state=seed).fit(X_train, y_train)

    probs = clf.predict_proba(X_test)[:, 1]

    y_pred = (probs > 0.5).astype(int)

    error = np.mean(y_pred != y_test)
    errors.append(error)

cv_test_error = np.mean(errors)

for i, err in enumerate(errors, 1):
    print(f"Fold {i}: Test Error = {err:.4f}")

print(f"\nEstimated 10-Fold CV Test Error: {cv_test_error:.4f}")


Fold 1: Test Error = 0.0270
Fold 2: Test Error = 0.0260
Fold 3: Test Error = 0.0290
Fold 4: Test Error = 0.0360
Fold 5: Test Error = 0.0200
Fold 6: Test Error = 0.0220
Fold 7: Test Error = 0.0240
Fold 8: Test Error = 0.0170
Fold 9: Test Error = 0.0330
Fold 10: Test Error = 0.0290

Estimated 10-Fold CV Test Error: 0.0263


# Part C

In [18]:
from sklearn.linear_model import LogisticRegression 

df_lr = pd.read_csv("Default.csv")
df_lr['student'] = df_lr['student'].map({'Yes': 1, 'No': 0})
df_lr['default'] = df_lr['default'].map({'Yes': 1, 'No': 0})

X_lr = df_lr[['income', 'balance', 'student']].values
y_lr = df_lr['default']

kf_lr = KFold(n_splits=10, shuffle=True, random_state=seed)
errors_lr = []

for train_index, test_index in kf_lr.split(X_lr):

    X_train_lr, X_test_lr = X_lr[train_index], X_lr[test_index]
    y_train_lr, y_test_lr = y_lr.iloc[train_index], y_lr.iloc[test_index]

    clf_lr = LogisticRegression(random_state=seed, max_iter=1000).fit(X_train_lr, y_train_lr)

    probs_lr = clf_lr.predict_proba(X_test_lr)[:, 1]
    y_pred_lr = (probs_lr > 0.5).astype(int)

    error_lr = np.mean(y_pred_lr != y_test_lr)
    errors_lr.append(error_lr)

cv_test_error_lr = np.mean(errors_lr)
for i, err in enumerate(errors_lr, 1):
    print(f"Fold {i}: Test Error = {err:.4f}")

print(f"\nEstimated 10-Fold CV Test Error with Student Variable: {cv_test_error_lr:.4f}")

Fold 1: Test Error = 0.0290
Fold 2: Test Error = 0.0270
Fold 3: Test Error = 0.0290
Fold 4: Test Error = 0.0370
Fold 5: Test Error = 0.0200
Fold 6: Test Error = 0.0230
Fold 7: Test Error = 0.0240
Fold 8: Test Error = 0.0180
Fold 9: Test Error = 0.0360
Fold 10: Test Error = 0.0300

Estimated 10-Fold CV Test Error with Student Variable: 0.0273


Before including a dummy variable for student in the 10-fold cross-validation, the test error rate of the model was 0.0263. After including the dummy variable for student, the 10 fold cross validation test error rate slightly rose to 0.0273. This 0.001 difference is extremely miniscule, but overall tells us that the dummy student variable doesn't lead to a reduction in test error rate. Therefore doesn't provide any meaningful insight into the output. 

# Part D

In [ ]:
df_xg = pd.read_csv("Default.csv")
df_xg['student'] = df_xg['student'].map({'Yes': 1, 'No': 0})
X = df_xg[['income', 'balance', 'student']].values
df_xg['default'] = df_xg['default'].map({'Yes': 1, 'No': 0})
y = df_xg['default']

import xgboost as xgb
errors = []

for train_index, test_index in kf.split(X):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    clf = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=seed)
    clf.fit(X_train, y_train)

    probs = clf.predict_proba(X_test)[:, 1]

    y_pred = (probs > 0.5).astype(int)

    error = np.mean(y_pred != y_test)
    errors.append(error)

cv_test_error = np.mean(errors)

for i, err in enumerate(errors, 1):
    print(f"Fold {i}: Test Error = {err:.4f}")

print(f"\nEstimated 10-Fold CV XGBoost Test Error: {cv_test_error:.4f}")

Fold 1: Test Error = 0.0280
Fold 2: Test Error = 0.0350
Fold 3: Test Error = 0.0330
Fold 4: Test Error = 0.0430
Fold 5: Test Error = 0.0200
Fold 6: Test Error = 0.0260
Fold 7: Test Error = 0.0360
Fold 8: Test Error = 0.0200
Fold 9: Test Error = 0.0350
Fold 10: Test Error = 0.0320

Estimated 10-Fold CV XGBoost Test Error: 0.0308


Using XGBoost instead of logistic regression raised the test error from 0.0273 to 0.0308. The change is about 0.03, which is a relatively small change, indicating that using XGBoost instead of regression does not cause a big difference. The slight raise in error might be explained by the data following more of a linear relationship, or that the default XGBoost parameters are not optimized for this data.